# Golden set workbench

Write `evals/golden.jsonl` here. Everything in this notebook **reads data only**: the contracts, their CUAD spans, the Item 1A text, the revenue table and the playbook. It never touches the index, the retriever or an embedder, so using it is not a retrieval run.

Format and mix: `evals/GOLDEN.md` (10 clause, 6 filing, 7 filter, 7 aggregate with 3+ on revenue, 5 playbook_check with 1+ cross-document, 5 unanswerable).

Run the cells top to bottom. Section 6 builds questions, section 7 saves and checks them; saving keeps what is already in the file, so you can stop and come back.

In [ ]:
# ── IMPORTS ─────────────────────────────────────────────
import sys                      # NOTE: put the repo on the import path
import json                     # NOTE: read and write golden.jsonl
import importlib.util           # NOTE: load evals/check_golden.py
from datetime import datetime   # NOTE: timestamps in the status lines
from pathlib import Path        # NOTE: file paths
import pandas as pd             # NOTE: tables to browse

# ── REPO ROOT ───────────────────────────────────────────
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "golddata.py").exists())
sys.path.insert(0, str(ROOT))
from core import golddata as G  # NOTE: read-only data access (no retriever, no embedder)

_spec = importlib.util.spec_from_file_location("check_golden", ROOT / "evals" / "check_golden.py")
CG = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(CG)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

def stamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# OUTPUT → should print the repo path and a ✓ with no errors
print(f"[{stamp()}] repo: {ROOT}")
print("✓ All packages loaded")

In [ ]:
# ── LOAD DATA ───────────────────────────────────────────
print(f"[{stamp()}] loading the selected contracts, EDGAR outputs and playbook")
records   = {r["id"]: r for r in G.selected_records()}       # NOTE: the 80 selected contracts
accounts  = G.accounts()                                      # NOTE: contract id → account, CIK
risk      = {e["doc_id"]: e for e in G.risk_entries()}        # NOTE: Item 1A texts, doc id rf_<cik>_<accession>
revenue   = pd.DataFrame(G.revenue_rows())                    # NOTE: one row per company and fiscal year
playbook  = G.playbook_text()

# OUTPUT → should print 80 contracts and non-zero Item 1A and revenue counts
print(f"  contracts: {len(records)} · Item 1A sections: {len(risk)} · revenue rows: {len(revenue)} "
      f"({revenue['cik'].nunique() if len(revenue) else 0} companies)")
print(f"✓ [{stamp()}] data loaded")

## 1. The contracts and their parsed metadata

Only parsed values appear; a blank means the label did not parse cleanly, so filter and aggregate questions should use only filled columns.

In [ ]:
# ── CONTRACT TABLE ──────────────────────────────────────
rows = []
for cid, r in records.items():
    a = accounts.get(cid, {})
    rows.append({"contract_id": cid, "account": a.get("account"), "cik": a.get("cik"), "type": r["contract_type"],
                 **{f: G.parsed(r, f) for f in G.META_FIELDS},
                 "clause_categories": len(G.clause_categories(r)), "chars": len(r["text"])})
contracts = pd.DataFrame(rows).sort_values(["type", "account"]).reset_index(drop=True)

# OUTPUT → should show 80 rows
print(f"✓ [{stamp()}] {len(contracts)} contracts")
contracts

In [ ]:
# ── WHAT FILTER AND AGGREGATE QUESTIONS CAN USE ─────────
contracts["agreement_year"] = contracts["agreement_date"].str[:4]
for col in ["type", "governing_law", "agreement_year", "renewal_term", "notice_period_to_terminate_renewal", "expiration_date"]:
    counts = contracts[col].value_counts(dropna=True)
    print(f"\n{col}: {counts.sum()} of {len(contracts)} parsed")
    print(counts.head(12).to_string())

# OUTPUT → should list the value counts per field
print(f"\n✓ [{stamp()}] metadata summarized")

## 2. Look-up helpers

Offsets are `[start, end)` into the text exactly as the pipeline loads it, so they go straight into `gold_passages`.

In [ ]:
# ── HELPERS ─────────────────────────────────────────────
def one_line(s):
    return " ".join(s.split())

def spans(contract_id, category=None):
    """CUAD's lawyer-labelled spans for a contract, optionally one category."""
    r = records[contract_id]
    out = [{"category": s["category"], "start": s["start"], "end": s["end"], "text": one_line(r["text"][s["start"]:s["end"]])}
           for s in r["spans"] if not category or s["category"].lower() == category.lower()]
    return pd.DataFrame(out)

def _hits(text, phrase, doc_id, width=80):
    out = []
    for s, e in G.find(text, phrase):
        out.append({"doc_id": doc_id, "start": s, "end": e, "match": one_line(text[s:e]),
                    "context": "…" + one_line(text[max(0, s - width):s]) + " [[" + one_line(text[s:e]) + "]] " + one_line(text[e:e + width]) + "…"})
    return out

def find(contract_id, phrase):
    """Offsets of a phrase in a contract (any spacing, any case)."""
    return pd.DataFrame(_hits(records[contract_id]["text"], phrase, contract_id))

def risk_find(cik, phrase):
    """Offsets of a phrase in the company's extracted Item 1A."""
    out = []
    for doc_id, e in risk.items():
        if int(e["cik"]) == int(cik):
            out += _hits(e["text"], phrase, doc_id)
    return pd.DataFrame(out)

def playbook_find(phrase):
    """Offsets of a phrase in data/playbook.md (doc id 'playbook')."""
    return pd.DataFrame(_hits(playbook, phrase, "playbook"))

def revenue_for(cik):
    return revenue[revenue["cik"] == int(cik)][["row_id", "fiscal_year_end", "revenue_usd", "concept"]] if len(revenue) else revenue

def contracts_for(**where):
    """Contracts whose parsed metadata equals every value given, e.g. contracts_for(governing_law="Delaware")."""
    df = contracts
    for k, v in where.items():
        df = df[df[k] == v]
    return df

# OUTPUT → should print a ✓
print(f"✓ [{stamp()}] helpers ready: spans, find, risk_find, playbook_find, revenue_for, contracts_for")

In [ ]:
# ── EXAMPLE: A CLAUSE AND ITS GOLD SPAN ─────────────────
example = next(cid for cid, r in records.items() if any(s["category"] == "Cap On Liability" for s in r["spans"]))
print(f"[{stamp()}] {example} · {accounts.get(example, {}).get('account')} · {records[example]['contract_type']}")

# OUTPUT → should show the Cap On Liability spans for one contract
spans(example, "Cap On Liability")

## 3. Build questions

Each builder returns one line in the `evals/GOLDEN.md` format. `add()` checks it on the spot against the data and prints any problem; fix and re-run that cell. Re-adding a question with the same id replaces it.

In [ ]:
# ── QUESTION BUILDERS ───────────────────────────────────
GOLDEN_PATH = ROOT / "evals" / "golden.jsonl"
GOLDEN = {}
if GOLDEN_PATH.exists():  # resume where you left off
    for line in GOLDEN_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            q = json.loads(line)
            GOLDEN[q["id"]] = q
SOURCES = CG.load_sources()

def _q(id, kind, question, *, scope=None, passages=(), rows=(), contracts=(), facts=(), verdict=None, notes=""):
    return {"id": id, "kind": kind, "question": question, "scope": dict(scope or {}),
            "gold_passages": [dict(p) for p in passages], "gold_rows": list(rows), "expected_contract_ids": list(contracts),
            "expected_facts": list(facts), "expected_verdict": verdict, "notes": notes}

def passage(doc_id, start, end, category=None):
    p = {"doc_id": doc_id, "start": int(start), "end": int(end)}
    if category:
        p["category"] = category
    return p

def clause(id, question, contract_id, passages, facts, notes=""):
    return _q(id, "clause", question, scope={"contract_id": contract_id, "doc_type": "contract"}, passages=passages, facts=facts, notes=notes)

def filing(id, question, cik, passages, facts=(), notes=""):
    return _q(id, "filing", question, scope={"account": accounts_by_cik.get(int(cik)), "doc_type": "risk_factors"}, passages=passages, facts=facts, notes=notes)

def filter_q(id, question, contract_ids, notes=""):
    return _q(id, "filter", question, contracts=contract_ids, notes=notes)

def aggregate(id, question, facts, rows=(), contract_ids=(), notes=""):
    return _q(id, "aggregate", question, rows=rows, contracts=contract_ids, facts=facts, notes=notes)

def playbook_check(id, question, contract_id, passages, verdict, rows=(), notes=""):
    return _q(id, "playbook_check", question, scope={"contract_id": contract_id}, passages=passages, rows=rows, verdict=verdict, notes=notes)

def unanswerable(id, question, notes=""):
    return _q(id, "unanswerable", question, notes=notes)

accounts_by_cik = {int(a["cik"]): a["account"] for a in accounts.values() if a.get("cik")}

def add(q):
    problems = CG.check_line(dict(q), SOURCES)
    GOLDEN[q["id"]] = q
    if problems:
        print(f"✗ {q['id']} added with {len(problems)} problem(s):")
        for p in problems:
            print("   -", p)
    else:
        print(f"✓ {q['id']} ({q['kind']}) added · {len(GOLDEN)} in the set")

def mix():
    have = pd.Series([q["kind"] for q in GOLDEN.values()]).value_counts()
    return pd.DataFrame({"have": have, "need": pd.Series(CG.MIX)}).fillna(0).astype(int)

# OUTPUT → should print how many questions are already in golden.jsonl
print(f"✓ [{stamp()}] builders ready · {len(GOLDEN)} question(s) loaded from {GOLDEN_PATH.name if GOLDEN_PATH.exists() else 'nothing yet'}")
mix()

In [ ]:
# ── EXAMPLES (edit, copy, delete) ───────────────────────
# A clause question: pick a span from spans(...), use its start/end, and a fact that appears inside it.
s = spans(example, "Cap On Liability").iloc[0]
q = clause("C01", "What is the cap on liability in this agreement?", example,
           passages=[passage(example, s.start, s.end, "Cap On Liability")],
           facts=[s.text.split()[0]])          # NOTE: replace with a phrase a correct answer must contain
print(json.dumps(q, indent=1)[:600])
# add(q)                                       # NOTE: uncomment to add it

# OUTPUT → should print the question as JSON; nothing is added until add() runs

## 4. Save and check

Writes every question in `GOLDEN` to `evals/golden.jsonl` (sorted by id) and runs the full checker, including the mix. Commit once it prints 0 problems; the file is frozen from then on.

In [ ]:
# ── SAVE AND CHECK ──────────────────────────────────────
if not GOLDEN:
    problems = ["no questions yet: nothing written (an empty golden.jsonl would look like a finished one)"]
else:
    print(f"[{stamp()}] writing {len(GOLDEN)} question(s) to {GOLDEN_PATH.relative_to(ROOT)}")
    GOLDEN_PATH.write_text("".join(json.dumps(GOLDEN[k], ensure_ascii=False) + "\n" for k in sorted(GOLDEN)), encoding="utf-8")
    problems = CG.check(GOLDEN_PATH)
for p in problems:
    print("   -", p)

# OUTPUT → should end with ✓ and 0 problems once all 40 are written
print(("✓" if not problems else "✗") + f" [{stamp()}] {len(problems)} problem(s)")
mix()